# Analisis Sentimen Ulasan Produk Tokopedia 2025

Notebook mencakup pengambilan data, eksplorasi, preprocessing teks bahasa Indonesia,
pelatihan Logistic Regression, penanganan ketidakseimbangan kelas, serta eksperimen IndoBERT.


## 1. Persiapan lingkungan

In [ ]:
%pip install -q kagglehub Sastrawi wordcloud imbalanced-learn transformers datasets torch

In [ ]:
import random
import re
from pathlib import Path

import kagglehub
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.pipeline import Pipeline
from wordcloud import WordCloud

RANDOM_STATE = 42
LABELS = ["negative", "neutral", "positive"]
random.seed(RANDOM_STATE)
sns.set_theme(style="whitegrid")


## 2. Memuat dataset

In [ ]:
dataset_dir = Path(kagglehub.dataset_download("salmanabdu/tokopedia-product-reviews-2025"))
csv_path = next(dataset_dir.glob("*.csv"))
df = pd.read_csv(csv_path)

print(f"Dataset: {csv_path.name}")
print(f"Ukuran: {df.shape[0]:,} baris × {df.shape[1]} kolom")
display(df.head())

## 3. Eksplorasi data (EDA)


In [ ]:
df.info()
display(df.describe(include="all"))
display(df.isna().sum().rename("missing_values"))
display(df["rating"].value_counts().sort_index().rename("jumlah"))
display(df["sentiment_label"].value_counts().rename("jumlah"))

In [ ]:
plt.figure(figsize=(8, 5))
sns.countplot(
    data=df,
    x="sentiment_label",
    order=LABELS,
    hue="sentiment_label",
    palette="viridis",
    legend=False,
)
plt.title("Distribusi Label Sentimen")
plt.xlabel("Sentimen")
plt.ylabel("Jumlah Ulasan")
plt.show()

## 4. Preprocessing teks

In [ ]:
stopword_remover = StopWordRemoverFactory().create_stop_word_remover()

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"https?://\S+|www\.\S+", " ", text)
    text = re.sub(r"[^a-z\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return stopword_remover.remove(text)

sample_text = df["review_text"].dropna().iloc[0]
print("Sebelum:", sample_text)
print("Sesudah:", clean_text(sample_text))

In [ ]:
stemmer = StemmerFactory().create_stemmer()

def clean_text_advanced(text):
    return stemmer.stem(clean_text(text))

print(f"Sebelum stemming: {sample_text[:50]}...")
print(f"Sesudah stemming: {clean_text_advanced(sample_text)}")


### 4.1 Normalisasi slang dan emotikon

Bagian ini menambahkan kamus slang dan konversi emotikon untuk meningkatkan kualitas fitur teks.


In [ ]:
slang_dict = {
    "bgt": "banget", "bgtt": "banget", "jd": "jadi", "ga": "tidak",
    "gak": "tidak", "tdk": "tidak", "sdh": "sudah", "udah": "sudah",
    "brg": "barang", "krn": "karena", "cpt": "cepat", "jg": "juga",
    "tp": "tapi", "kalo": "kalau", "dgn": "dengan", "dr": "dari",
    "ori": "original", "mksh": "terima kasih", "tks": "terima kasih"
}

emoticon_map = {
    "😊": " senang ", "🙂": " senang ", "😄": " senang ", "😍": " suka ",
    "👍": " bagus ", "👌": " bagus ", "⭐": " bintang ",
    "😞": " kecewa ", "😟": " kecewa ", "😠": " marah ", "😡": " marah ",
    "👎": " buruk ", "😢": " sedih ", "😭": " sedih "
}

def normalize_slang(text):
    words = text.split()
    return " ".join([slang_dict.get(w, w) for w in words])

def convert_emoticons(text):
    for emo, replacement in emoticon_map.items():
        text = text.replace(emo, replacement)
    return text

def clean_text_final(text):
    # Konversi emotikon sebelum regex menghapusnya
    text = convert_emoticons(str(text))
    text = clean_text(text)
    text = normalize_slang(text)
    return stemmer.stem(text)

test_slang = "barangnya bgt bagus, tp pengiriman agak lama jd kecewa 😊"
print(f"Original: {test_slang}")
print(f"Processed: {clean_text_final(test_slang)}")

## 5. Pelatihan model baseline

In [ ]:
n_samples = 700
balanced_parts = []

for label in LABELS:
    group = df[df["sentiment_label"] == label]
    balanced_parts.append(
        group.sample(
            n=n_samples,
            replace=len(group) < n_samples,
            random_state=RANDOM_STATE,
        )
    )

df_balanced = pd.concat(balanced_parts, ignore_index=True)
df_balanced["clean_text"] = df_balanced["review_text"].apply(clean_text)

X_train, X_test, y_train, y_test = train_test_split(
    df_balanced["clean_text"],
    df_balanced["sentiment_label"],
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=df_balanced["sentiment_label"],
)

baseline_tfidf = TfidfVectorizer(max_features=2_000)
X_train_tfidf = baseline_tfidf.fit_transform(X_train)
X_test_tfidf = baseline_tfidf.transform(X_test)

baseline_model = LogisticRegression(max_iter=1_000, random_state=RANDOM_STATE)
baseline_model.fit(X_train_tfidf, y_train)

## 6. Evaluasi model baseline

In [ ]:
y_pred = baseline_model.predict(X_test_tfidf)
print(classification_report(y_test, y_pred, labels=LABELS, zero_division=0))

cm = confusion_matrix(y_test, y_pred, labels=LABELS)
plt.figure(figsize=(8, 6))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=LABELS,
    yticklabels=LABELS,
)
plt.xlabel("Prediksi")
plt.ylabel("Aktual")
plt.title("Confusion Matrix")
plt.show()

## 7. Optimasi hyperparameter

In [ ]:
df_balanced["clean_text"] = df_balanced["review_text"].apply(clean_text_advanced)

X_train, X_test, y_train, y_test = train_test_split(
    df_balanced["clean_text"],
    df_balanced["sentiment_label"],
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=df_balanced["sentiment_label"],
)

pipeline = Pipeline(
    [
        ("tfidf", TfidfVectorizer()),
        ("clf", LogisticRegression(max_iter=1_000, random_state=RANDOM_STATE)),
    ]
)
param_grid = {
    "tfidf__max_features": [1_000, 2_000, 3_000],
    "tfidf__ngram_range": [(1, 1), (1, 2)],
    "clf__C": [0.1, 1, 10],
}

grid_search = GridSearchCV(
    pipeline,
    param_grid,
    cv=5,
    n_jobs=-1,
    scoring="f1_macro",
)
grid_search.fit(X_train, y_train)

print(f"Parameter terbaik: {grid_search.best_params_}")
print(f"Skor CV terbaik: {grid_search.best_score_:.4f}")


In [ ]:
best_model = grid_search.best_estimator_
y_pred_tuned = best_model.predict(X_test)

print("Hasil setelah tuning")
print(
    classification_report(
        y_test,
        y_pred_tuned,
        labels=LABELS,
        zero_division=0,
    )
)


## 8. Visualisasi kata dominan

In [ ]:
def show_wordcloud(sentiment, colormap):
    text = " ".join(
        df_balanced.loc[
            df_balanced["sentiment_label"] == sentiment, "clean_text"
        ]
    )
    wordcloud = WordCloud(
        width=800,
        height=400,
        background_color="white",
        colormap=colormap,
        max_words=100,
    ).generate(text)

    plt.figure(figsize=(10, 5))
    plt.imshow(wordcloud, interpolation="bilinear")
    plt.axis("off")
    plt.title(f"Kata Dominan: {sentiment.upper()}")
    plt.show()

for sentiment, colormap in zip(LABELS, ["Reds", "plasma", "viridis"]):
    show_wordcloud(sentiment, colormap)

## 9. Prediksi ulasan baru

In [ ]:
def prediksi_sentimen(teks):
    teks_bersih = clean_text(teks)
    vektor = baseline_tfidf.transform([teks_bersih])
    return baseline_model.predict(vektor)[0]


contoh_ulasan = [
    "Barangnya rusak saat sampai",
    "Bagus banget, pengiriman cepat",
    "Biasa saja, lumayan",
]

for ulasan in contoh_ulasan:
    print(f"{ulasan!r}: {prediksi_sentimen(ulasan)}")

## 10. Penanganan ketidakseimbangan kelas

Bagian ini membandingkan augmentasi teks, SMOTE, dan class weighting.

### 10.1 Augmentasi teks

In [ ]:

synonym_dict = {
    "bagus": ["baik", "oke", "mantap", "keren"],
    "buruk": ["jelek", "kurang", "parah"],
    "kecewa": ["kesal", "nyesel", "sedih"],
    "cepat": ["kilat", "gegas", "ekspres"],
    "lama": ["lambat", "lelet", "telat"]
}

def augment_text(text):
    words = text.split()
    new_words = words.copy()
    for i, word in enumerate(words):
        if word in synonym_dict:
            if random.random() > 0.5:
                new_words[i] = random.choice(synonym_dict[word])
    return " ".join(new_words)

neg_samples = df[df['sentiment_label'] == 'negative']['review_text'].iloc[:2]
for txt in neg_samples:
    print(f"Original: {txt[:50]}...")
    print(f"Augmented: {augment_text(txt)[:50]}...")

### 10.2 SMOTE

`imbalanced-learn` menyeimbangkan fitur TF-IDF secara sintetis.


In [ ]:
from imblearn.over_sampling import SMOTE

df_smote = df.sample(n=min(5_000, len(df)), random_state=RANDOM_STATE).copy()
df_smote['clean_text'] = df_smote['review_text'].apply(clean_text_final)

tfidf_smote = TfidfVectorizer(max_features=2000)
X_tfidf = tfidf_smote.fit_transform(df_smote['clean_text'])
y = df_smote['sentiment_label']

smote = SMOTE(random_state=RANDOM_STATE)
X_resampled, y_resampled = smote.fit_resample(X_tfidf, y)

print("Jumlah data sebelum SMOTE:", y.value_counts().to_dict())
print("Jumlah data setelah SMOTE:", pd.Series(y_resampled).value_counts().to_dict())

In [ ]:
X_train_s, X_test_s, y_train_s, y_test_s = train_test_split(
    X_resampled, y_resampled, test_size=0.2, random_state=RANDOM_STATE
)

smote_model = LogisticRegression(max_iter=1000)
smote_model.fit(X_train_s, y_train_s)

y_pred_s = smote_model.predict(X_test_s)
print("Laporan klasifikasi dengan SMOTE:")
print(classification_report(y_test_s, y_pred_s))

In [ ]:
cm_smote = confusion_matrix(y_test_s, y_pred_s, labels=LABELS)

plt.figure(figsize=(8, 6))
sns.heatmap(
    cm_smote,
    annot=True,
    fmt="d",
    cmap="Greens",
    xticklabels=LABELS,
    yticklabels=LABELS,
)
plt.xlabel("Prediksi")
plt.ylabel("Aktual")
plt.title("Confusion Matrix: Model SMOTE")
plt.show()


### 10.3 Class weighting

Eksperimen ini memakai subset data lebih besar dan `class_weight="balanced"` untuk
menangani ketidakseimbangan kelas tanpa undersampling.


In [ ]:
df_large = df.sample(n=min(10_000, len(df)), random_state=RANDOM_STATE).copy()
df_large["clean_text"] = df_large["review_text"].apply(clean_text)

X_train_large, X_test_large, y_train_large, y_test_large = train_test_split(
    df_large["clean_text"],
    df_large["sentiment_label"],
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=df_large["sentiment_label"],
)

weighted_model = Pipeline(
    [
        ("tfidf", TfidfVectorizer(max_features=3_000)),
        (
            "clf",
            LogisticRegression(
                max_iter=1_000,
                class_weight="balanced",
                random_state=RANDOM_STATE,
            ),
        ),
    ]
)
weighted_model.fit(X_train_large, y_train_large)

y_pred_large = weighted_model.predict(X_test_large)
print("Laporan klasifikasi dengan class weighting:")
print(
    classification_report(
        y_test_large,
        y_pred_large,
        labels=LABELS,
        zero_division=0,
    )
)

### 10.4 Peningkatan deteksi kelas netral

Eksperimen menggabungkan fitur panjang teks, data latih seimbang, dan penyesuaian
probabilitas untuk memperkuat sinyal kelas `neutral`.

In [ ]:
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import FeatureUnion

import numpy as np


n_balanced = 500
balanced_train = pd.concat(
    [
        df[df["sentiment_label"] == label].sample(
            n=min(n_balanced, (df["sentiment_label"] == label).sum()),
            random_state=RANDOM_STATE,
        )
        for label in LABELS
    ]
)

X_train_bal = balanced_train["review_text"].apply(clean_text)
y_train_bal = balanced_train["sentiment_label"]


class TextStats(BaseEstimator, TransformerMixin):
    def fit(self, texts, labels=None):
        return self

    def transform(self, texts):
        return np.array([len(str(text)) for text in texts]).reshape(-1, 1)


enhanced_pipeline = Pipeline(
    [
        (
            "features",
            FeatureUnion(
                [
                    (
                        "tfidf",
                        TfidfVectorizer(max_features=3_000, ngram_range=(1, 2)),
                    ),
                    ("stats", TextStats()),
                ]
            ),
        ),
        (
            "clf",
            LogisticRegression(
                class_weight="balanced",
                max_iter=1_000,
                random_state=RANDOM_STATE,
            ),
        ),
    ]
)
enhanced_pipeline.fit(X_train_bal, y_train_bal)

y_probs = enhanced_pipeline.predict_proba(X_test_large)
neutral_index = list(enhanced_pipeline.classes_).index("neutral")
y_probs[:, neutral_index] += 0.2
y_pred_tuned = enhanced_pipeline.classes_[np.argmax(y_probs, axis=1)]

print("Laporan klasifikasi (balanced training + threshold tuning):")
print(
    classification_report(
        y_test_large,
        y_pred_tuned,
        labels=LABELS,
        zero_division=0,
    )
)

In [ ]:
cm_enhanced = confusion_matrix(y_test_large, y_pred_tuned, labels=LABELS)

plt.figure(figsize=(8, 6))
sns.heatmap(
    cm_enhanced,
    annot=True,
    fmt="d",
    cmap="Purples",
    xticklabels=LABELS,
    yticklabels=LABELS,
)
plt.xlabel("Prediksi")
plt.ylabel("Aktual")
plt.title("Confusion Matrix: Enhanced Model (Balanced + Threshold Tuning)")
plt.show()

In [ ]:
positive_as_neutral_mask = (
    (y_test_large.to_numpy() == "positive")
    & (np.asarray(y_pred_tuned) == "neutral")
)
positive_as_neutral = pd.DataFrame(
    {
        "review_text": X_test_large.to_numpy()[positive_as_neutral_mask],
        "sentiment_label": y_test_large.to_numpy()[positive_as_neutral_mask],
    }
)

print(f"Total positive diprediksi neutral: {len(positive_as_neutral)}")
print("-" * 30)

for _, row in positive_as_neutral.head(10).iterrows():
    print(f"Ulasan: {row['review_text']}")
    print(f"Sentimen asli: {row['sentiment_label']}")
    print("-" * 10)

## 11. Eksperimen IndoBERT

Bagian opsional ini memakai `indobenchmark/indobert-base-p2`. Jalankan dengan runtime
GPU agar fine-tuning lebih cepat.

In [ ]:
from collections import Counter
from glob import glob
import os

import numpy as np
import torch
from datasets import Dataset
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    Trainer,
    TrainingArguments,
)

BERT_MODEL_NAME = "indobenchmark/indobert-base-p2"
tokenizer = AutoTokenizer.from_pretrained(BERT_MODEL_NAME)

label2id = {label: index for index, label in enumerate(LABELS)}
id2label = {index: label for index, label in enumerate(LABELS)}

bert_model = AutoModelForSequenceClassification.from_pretrained(
    BERT_MODEL_NAME,
    num_labels=len(LABELS),
    id2label=id2label,
    label2id=label2id,
)


def tokenize_function(examples):
    return tokenizer(
        examples["review_text"],
        padding="max_length",
        truncation=True,
        max_length=128,
    )

### 11.1 Menyiapkan dataset

In [ ]:
train_df, val_df = train_test_split(
    df_balanced,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=df_balanced["sentiment_label"],
)

train_dataset = Dataset.from_pandas(
    train_df[["review_text", "sentiment_label"]],
    preserve_index=False,
)
val_dataset = Dataset.from_pandas(
    val_df[["review_text", "sentiment_label"]],
    preserve_index=False,
)

train_dataset = train_dataset.map(
    lambda row: {"labels": label2id[row["sentiment_label"]]}
)
val_dataset = val_dataset.map(
    lambda row: {"labels": label2id[row["sentiment_label"]]}
)
train_dataset = train_dataset.map(tokenize_function, batched=True)
val_dataset = val_dataset.map(tokenize_function, batched=True)

### 11.2 Fine-tuning

Konfigurasi berikut melatih model selama tiga epoch dan menyimpan model terbaik.

In [ ]:
def compute_metrics(prediction):
    labels = prediction.label_ids
    predictions = prediction.predictions.argmax(axis=-1)
    precision, recall, f1, _ = precision_recall_fscore_support(
        labels,
        predictions,
        average="macro",
        zero_division=0,
    )
    return {
        "accuracy": accuracy_score(labels, predictions),
        "f1": f1,
        "precision": precision,
        "recall": recall,
    }


training_args = TrainingArguments(
    output_dir="results",
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    warmup_steps=100,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    logging_steps=10,
    report_to="none",
)

trainer = Trainer(
    model=bert_model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
)
trainer.train()

### 11.3 Evaluasi IndoBERT

In [ ]:
bert_metrics = trainer.evaluate()
bert_metrics

In [ ]:
prediction_output = trainer.predict(val_dataset)
y_pred_bert = np.argmax(prediction_output.predictions, axis=-1)
y_true_bert = prediction_output.label_ids

cm_bert = confusion_matrix(
    y_true_bert,
    y_pred_bert,
    labels=range(len(LABELS)),
)

plt.figure(figsize=(8, 6))
sns.heatmap(
    cm_bert,
    annot=True,
    fmt="d",
    cmap="Oranges",
    xticklabels=LABELS,
    yticklabels=LABELS,
)
plt.xlabel("Prediksi")
plt.ylabel("Aktual")
plt.title("Confusion Matrix: IndoBERT")
plt.show()


### 11.4 Analisis kesalahan

Lima ulasan berikut merupakan contoh prediksi IndoBERT yang keliru.


In [ ]:
error_indices = np.where(y_pred_bert != y_true_bert)[0]
error_samples = []

for index in error_indices[:5]:
    error_samples.append(
        {
            "Ulasan": val_df.iloc[index]["review_text"],
            "Aktual": id2label[y_true_bert[index]],
            "Prediksi": id2label[y_pred_bert[index]],
        }
    )

df_errors = pd.DataFrame(error_samples)
display(df_errors)


### 11.5 Visualisasi bobot attention

Visualisasi berikut menunjukkan pola attention IndoBERT untuk satu ulasan contoh.


In [ ]:
checkpoints = glob("results/checkpoint-*")
last_checkpoint = (
    max(checkpoints, key=os.path.getctime) if checkpoints else BERT_MODEL_NAME
)
print(f"Memuat model dari: {last_checkpoint}")

attention_model = AutoModelForSequenceClassification.from_pretrained(
    last_checkpoint,
    num_labels=len(LABELS),
    id2label=id2label,
    label2id=label2id,
    attn_implementation="eager",
).to(trainer.args.device)


def visualize_attention(text):
    inputs = tokenizer(
        text,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=128,
    ).to(attention_model.device)

    with torch.no_grad():
        outputs = attention_model(**inputs, output_attentions=True)

    attention = outputs.attentions[-1][0, 0].cpu().numpy()
    tokens = tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])

    plt.figure(figsize=(10, 8))
    sns.heatmap(
        attention[: len(tokens), : len(tokens)],
        xticklabels=tokens,
        yticklabels=tokens,
        cmap="YlGnBu",
    )
    plt.title(f"Attention Map\nUlasan: {text[:50]!r}")
    plt.xticks(rotation=45)
    plt.show()


sample_review = "barangnya rusak parah dan pengiriman sangat lambat"
visualize_attention(sample_review)


### 11.6 Perbandingan model


In [ ]:
precision_smote, recall_smote, f1_smote, _ = precision_recall_fscore_support(
    y_test_s,
    y_pred_s,
    average="macro",
    zero_division=0,
)

comparison_data = {
    "Metrik": ["Accuracy", "Precision macro", "Recall macro", "F1-score macro"],
    "SMOTE (Logistic Regression)": [
        accuracy_score(y_test_s, y_pred_s),
        precision_smote,
        recall_smote,
        f1_smote,
    ],
    "IndoBERT": [
        bert_metrics["eval_accuracy"],
        bert_metrics["eval_precision"],
        bert_metrics["eval_recall"],
        bert_metrics["eval_f1"],
    ],
}

df_comparison = pd.DataFrame(comparison_data)
display(df_comparison.set_index("Metrik").style.format("{:.2%}"))


### 11.7 Pengujian IndoBERT pada salah klasifikasi positif sebagai netral

IndoBERT diuji pada ulasan positif yang sebelumnya diprediksi sebagai kelas
`neutral` oleh Logistic Regression.

In [ ]:
misclassified_texts = positive_as_neutral["review_text"].tolist()
misclassified_dataset = Dataset.from_dict({"review_text": misclassified_texts})


def tokenize_test(examples):
    return tokenizer(
        examples["review_text"],
        padding="max_length",
        truncation=True,
        max_length=128,
    )


misclassified_tokenized = misclassified_dataset.map(tokenize_test, batched=True)
bert_predictions = trainer.predict(misclassified_tokenized)
y_pred_bert_indices = np.argmax(bert_predictions.predictions, axis=-1)
y_pred_bert_labels = [id2label[index] for index in y_pred_bert_indices]

counts = Counter(y_pred_bert_labels)
print(f"Total sampel diuji: {len(misclassified_texts)}")
print("Hasil prediksi IndoBERT:")
for label, count in counts.items():
    print(f"- {label}: {count} ({count / len(misclassified_texts):.2%})")

print("\nContoh ulasan yang diperbaiki IndoBERT:")
correct_indices = [
    index
    for index, label in enumerate(y_pred_bert_labels)
    if label == "positive"
][:5]
for index in correct_indices:
    print(f"Ulasan: {misclassified_texts[index]}")